# Analysis for Removal of Raw Data

This notebook performs a basic analysis of the data created by `data_removal_raw.py`.

Because that analysis deletes portions of the signal based on fractions of a hearbeat
away from peak, we need to have some way to plot the original heartbeat on the same
scale. We do this by mapping each pixel to a corresponding "location" that is the fraction
of a heartbeat away from the nearest peak. We then use smoothing to create a single
heartbeat pattern.

In [ ]:
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
sys.path.append("../")
from scripts.constants import (
    DATA_DIR,
    MODEL_DIR,
    OUTPUT_DIR,
    N_LEADS,
)


In [ ]:
keep_only_retain_subjects = True

In [ ]:
# Generate this file by running ../scripts/data_removal_raw.py with `window_pct = 0.05` and `by_pct = 0.025`
removal = pd.read_csv('../output/removal_data_repl_interpol_raw/all_subjects_and_pixels_win50_inc25.csv')

In [ ]:
removal.head()

In [ ]:
filename = f"{DATA_DIR}/exams_part16.hdf5"

with h5py.File(filename, "r") as f:
    print(f"Keys in the HDF5 file: {list(f.keys())}")
    dataset = f['tracings']
    print(f"Dataset shape: {dataset.shape}")
    print(f"Dataset dtype:, {dataset.dtype}")
    data_array = f['tracings'][()]
    exam_ids = f['exam_id'][()]


In [ ]:
# Brute force sort df to match order of exam_ids
metadata = pd.read_csv(f'{DATA_DIR}/exams.csv')
metadata = metadata[metadata['trace_file'] == 'exams_part16.hdf5']

metadata_sorted = []
for exam_id in exam_ids:
    metadata_sorted.append(metadata[metadata['exam_id'] == exam_id])
metadata = pd.concat(metadata_sorted)
metadata['subject'] = np.arange(metadata.shape[0])

In [ ]:
beats_summary = pd.read_csv(f"{DATA_DIR}/beats_summary_frame.csv")

In [ ]:
# Get information on which subjects to retain
retain_subjects = beats_summary.groupby('subject')['retain_subject'].first().reset_index()
retain_subjects = retain_subjects[:20000]
retain_subjects = retain_subjects[retain_subjects['retain_subject'] == True]

# Limit to 20k observations
data_array = data_array[:20000, :, :]

if keep_only_retain_subjects:
    data_array = data_array[retain_subjects['subject'].values, :, :]
    metadata = metadata[metadata['subject'].isin(retain_subjects['subject'].values)]
    beats_summary = beats_summary[beats_summary['subject'].isin(retain_subjects['subject'].values)]


In [ ]:
use_closest_peak = False
peak_shift = 0.2  # If you're not mapping locations to the closest peak, you need to say
                  # what peak you're mapping to. This peak_shift parameter controls that
                  # relationship. As an example, setting peak_shift to 0.1 means that
                  # the closest peak will be the one that is closest to peaks - 0.1. Setting
                  # this parameter to a positive value means that you'll see less data
                  # before the QRS complex and more data after the QRS complex.

location_array = np.zeros_like(data_array)
for subject in range(data_array.shape[0]):
    if subject % 100 == 0:
        print(f"Subject {subject} of {data_array.shape[0]}")

    subject_number = retain_subjects.iloc[subject]['subject']

    for channel in range(data_array.shape[2]):

        # Extract the peaks
        peaks = beats_summary.loc[(beats_summary['subject'] == subject_number)
                                    & (beats_summary['channel'] == channel), 'peaks'].values[0]
        peaks = [int(item) for item in peaks.replace("[", "").replace("]", "").split()]

        # We can only calculate beat length if there are at least 2 peaks.
        if len(peaks) >= 2:
            avg_beat_length = (peaks[-1] - peaks[0]) / (len(peaks) - 1)
            for pixel in range(data_array.shape[1]):
                if use_closest_peak:
                    closest_peak = np.argmin([abs(pixel - i) for i in peaks])
                else:
                    closest_peak = np.argmin([abs(pixel - (i - peak_shift * avg_beat_length)) for i in peaks])
                location_array[subject, pixel, channel] = (
                    (pixel - peaks[closest_peak]) / avg_beat_length
                )


In [ ]:
# Check the mapping of pixels to peaks
plt.plot(location_array[0, :, 0])
plt.xlabel("Pixel location")
plt.ylabel("Distance (in avg_beat_lenght units) to mapped peak")
plt.show()

In [ ]:
# Reshape the arrays so that you have one column per channel.
x = np.reshape(location_array, shape=(location_array.shape[0] * location_array.shape[1], location_array.shape[2]))
y = np.reshape(data_array, shape=(data_array.shape[0] * data_array.shape[1], data_array.shape[2]))

In [ ]:
# This is a basic kernel smoother. It uses a normal kernel with standard deviation
# of kernel_width to smooth the series y and return the smoothed values at x.
def smooth(x, y, kernel_width=0.5):
    y_out = np.zeros_like(y)
    for i in range(len(y_out)):
        y_out[i] = np.average(y, weights=np.exp(-0.5 * (x - x[i])**2 / kernel_width))
    return y_out

In [ ]:
# For the plot, randomly select 10_000 points and plot the x values
# against the smoothed y values.
channel = 0
random_points = np.random.choice(x.shape[0], size=10_000, replace=False)
plot_x = x[random_points, channel]
plot_y = smooth(x[random_points, channel], y[random_points, channel], kernel_width=0.0005)
plt.scatter(plot_x[(plot_x > -0.5) & (plot_x < 0.5)],
            plot_y[(plot_x > -0.5) & (plot_x < 0.5)])
plt.show()

In [ ]:
# Calculate RMSE for each bin.
removal['center_pct'] = (removal['start_pct'] + removal['end_pct']) / 2
removal['sq_error'] = (removal['nn_predicted_age'] - removal['torch_pred'])**2
mse = removal.groupby('center_pct')[['sq_error', 'area_removed']].mean().reset_index()
mse['rmse'] = np.sqrt(mse['sq_error'])
mse

In [ ]:
# Plot the smoothed heartbeat and the RMSE by pixel location
fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('Pixel Location Relative to Peak')
ax1.set_ylabel('RMSE', color=color)
ax1.plot(mse['center_pct'], mse['rmse'], color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

color = 'tab:green'
ax2.set_ylabel(f'Smoothed raw signal', color=color)  # we already handled the x-label with ax1
ax2.scatter(plot_x[(plot_x > -0.5) & (plot_x < 0.5)],
            plot_y[(plot_x > -0.5) & (plot_x < 0.5)], 
            color=color, 
            alpha=0.01)
ax2.tick_params(axis='y', labelcolor=color)

# fig.tight_layout()  # otherwise the right y-label is slightly clipped
plt.title(f"")
# plt.savefig(f"{results_dir}/n{n_subjects}_s{pixels}.png")
plt.show()

In [ ]:
mse['area_adjusted_rmse'] = mse['rmse'] / mse['area_removed']

In [ ]:
# Plot the smoothed heartbeat and the area-adjusted RMSE by pixel location
fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('Pixel Location Relative to Peak')
ax1.set_ylabel('Area-Adjusted RMSE', color=color)
ax1.plot(mse['center_pct'], mse['area_adjusted_rmse'], color=color)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

color = 'tab:green'
ax2.set_ylabel(f'Smoothed raw signal', color=color)  # we already handled the x-label with ax1
ax2.scatter(plot_x[(plot_x > -0.5) & (plot_x < 0.5)],
            plot_y[(plot_x > -0.5) & (plot_x < 0.5)], 
            color=color, 
            alpha=0.01)
ax2.tick_params(axis='y', labelcolor=color)

# fig.tight_layout()  # otherwise the right y-label is slightly clipped
plt.title(f"")
# plt.savefig(f"{results_dir}/n{n_subjects}_s{pixels}.png")
plt.show()

In [ ]:
# Look at the relationship between area removed and RMSE
plt.scatter(mse['area_removed'], mse['rmse'])
plt.show()

In [ ]:
# Zoom in on the lower left corner
plt.scatter(mse['area_removed'], mse['rmse'])
plt.xlim([0, 200])
plt.ylim([0, 3])
plt.show()

In [ ]:
# For dots in the lower left corner, separate the ones with higher RMSE for their area
# from the ones with lower RMSE for the same area.
# Plot the smoothed hearbeat and RMSE colored by their location on the RMSE vs. area chart.
mse['color'] = 'blue'
mse.loc[(mse['area_removed'] < 75) & (mse['area_removed'] > 25) & (mse['rmse'] > 1), 'color'] = 'cyan'
mse.loc[(mse['area_removed'] < 75) & (mse['area_removed'] > 25) & (mse['rmse'] < 1), 'color'] = 'orange'

fig, ax1 = plt.subplots()

color = 'tab:red'
ax1.set_xlabel('Pixel Location Relative to Peak')
ax1.set_ylabel('RMSE', color=color)
ax1.scatter(mse['center_pct'], mse['rmse'], color='blue')
for color_value in ('cyan', 'orange'):
    ax1.scatter(mse[mse['color'] == color_value]['center_pct'],
            mse[mse['color'] == color_value]['rmse'],
            color = color_value)
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()  # instantiate a second Axes that shares the same x-axis

color = 'tab:green'
ax2.set_ylabel(f'Smoothed raw signal', color=color)  # we already handled the x-label with ax1
ax2.scatter(plot_x[(plot_x > -0.5) & (plot_x < 0.5)],
            plot_y[(plot_x > -0.5) & (plot_x < 0.5)], 
            color=color, 
            alpha=0.005)
ax2.tick_params(axis='y', labelcolor=color)

# fig.tight_layout()  # otherwise the right y-label is slightly clipped
plt.title(f"")
# plt.savefig(f"{results_dir}/n{n_subjects}_s{pixels}.png")
plt.show()